# 02 Feature Engineering

This notebook documents the feature layer that connects the normalized Phase 1 outputs to the business-facing analysis in `03_business_analysis.ipynb`. The goal is to make the transformation logic explicit, auditable, and reusable.

The notebook focuses on three things:
- loading the Phase 1 normalized tables
- building title-level features used in Phase 2
- validating the engineered feature layer before business analysis

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data/processed/titles.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))
pd.options.display.float_format = lambda value: f'{value:,.3f}'

from src.feature_engineering import (
    build_country_view,
    build_genre_view,
    build_title_features,
    load_phase1_tables,
)
from src.utils import ensure_directory, save_dataframe

## Load Phase 1 Assets

Phase 1 left us with a normalized base table plus bridge tables. This is the right starting point for feature engineering because it avoids recomputing structure from raw comma-separated strings.

In [2]:
tables = load_phase1_tables(PROJECT_ROOT / 'data/processed')

phase1_inventory = pd.DataFrame(
    [
        {'table_name': name, 'row_count': len(frame), 'column_count': frame.shape[1]}
        for name, frame in tables.items()
    ]
).sort_values('table_name').reset_index(drop=True)

display(phase1_inventory)

,table_name,row_count,column_count
0,title_cast,44310,2
1,title_country,7179,2
2,title_director,4851,2
3,title_genre,13670,2
4,titles,6234,15


## Build the Title-Level Feature Layer

The key design choice in Phase 2 is to move from normalized entities to a title-level analytical layer. That gives the business analysis notebook a stable table with freshness, breadth, and country-scope fields already computed.

In [3]:
title_features = build_title_features(
    tables['titles'],
    tables['title_country'],
    tables['title_genre'],
)
country_view = build_country_view(title_features, tables['title_country'])
genre_view = build_genre_view(title_features, tables['title_genre'])

display(title_features[['show_id', 'title', 'type', 'release_year', 'date_added_year', 'country_count', 'genre_count', 'country_scope', 'release_to_add_lag', 'release_to_add_lag_clean']].head(8))

,show_id,title,type,release_year,date_added_year,country_count,genre_count,country_scope,release_to_add_lag,release_to_add_lag_clean
0,1005494,The Stranger,Movie,1946,2018,1,3,Single-country,72,72
1,1008581,Stripes,Movie,1981,2019,1,3,Single-country,38,38
2,1029730,Teenage Mutant Ninja Turtles: The Movie,Movie,1990,2020,0,2,Missing country,30,30
3,1064058,Tremors 2: Aftershocks,Movie,1995,2020,0,3,Missing country,25,25
4,1065372,The Trigger Effect,Movie,1996,2018,1,1,Single-country,22,22
5,1067876,True Grit,Movie,1969,2020,1,2,Single-country,51,51
6,1145882,Young Tiger,Movie,1973,2016,1,2,Single-country,43,43
7,1150871,Love Jones,Movie,1997,2019,1,3,Single-country,22,22


In [4]:
feature_dictionary = pd.DataFrame(
    [
        {'feature_name': 'country_count', 'level': 'title', 'definition': 'Number of normalized country tags attached to a title', 'business_use': 'Measure supply breadth and identify single-country versus multi-country titles'},
        {'feature_name': 'genre_count', 'level': 'title', 'definition': 'Number of normalized genre tags attached to a title', 'business_use': 'Measure category breadth and support co-occurrence analysis'},
        {'feature_name': 'release_to_add_lag', 'level': 'title', 'definition': 'Raw difference between year_added and release_year', 'business_use': 'Quantify how quickly titles enter the catalog after release'},
        {'feature_name': 'is_valid_release_to_add_lag', 'level': 'title', 'definition': 'Flag for non-negative release-to-add lag', 'business_use': 'Separate usable freshness signals from metadata anomalies'},
        {'feature_name': 'release_to_add_lag_clean', 'level': 'title', 'definition': 'Release-to-add lag with negative values removed', 'business_use': 'Primary freshness metric for Phase 2 reporting'},
        {'feature_name': 'country_scope', 'level': 'title', 'definition': 'Missing country, Single-country, or Multi-country title scope', 'business_use': 'Support geographic footprint and co-production analysis'},
        {'feature_name': 'is_recent_within_1y', 'level': 'title', 'definition': 'Title added within 1 year of release', 'business_use': 'Measure near-current catalog freshness'},
        {'feature_name': 'is_recent_within_3y', 'level': 'title', 'definition': 'Title added within 3 years of release', 'business_use': 'Primary recent-share freshness KPI'},
        {'feature_name': 'is_recent_within_5y', 'level': 'title', 'definition': 'Title added within 5 years of release', 'business_use': 'Broader freshness KPI for slower-moving categories'},
    ]
)

display(feature_dictionary)

,feature_name,level,definition,business_use
0,country_count,title,Number of normalized country tags attached to ...,Measure supply breadth and identify single-cou...
1,genre_count,title,Number of normalized genre tags attached to a ...,Measure category breadth and support co-occurr...
2,release_to_add_lag,title,Raw difference between year_added and release_...,Quantify how quickly titles enter the catalog ...
3,is_valid_release_to_add_lag,title,Flag for non-negative release-to-add lag,Separate usable freshness signals from metadat...
4,release_to_add_lag_clean,title,Release-to-add lag with negative values removed,Primary freshness metric for Phase 2 reporting
5,country_scope,title,"Missing country, Single-country, or Multi-coun...",Support geographic footprint and co-production...
6,is_recent_within_1y,title,Title added within 1 year of release,Measure near-current catalog freshness
7,is_recent_within_3y,title,Title added within 3 years of release,Primary recent-share freshness KPI
8,is_recent_within_5y,title,Title added within 5 years of release,Broader freshness KPI for slower-moving catego...


## QA Checks

Before the business analysis notebook uses this layer, we want to verify that title coverage is preserved, anomalies are visible, and bridge views line up with the source counts.

In [5]:
feature_qa = pd.DataFrame(
    [
        {'check_name': 'titles_row_count', 'value': len(title_features)},
        {'check_name': 'distinct_show_id', 'value': title_features['show_id'].nunique()},
        {'check_name': 'negative_lag_titles', 'value': int((title_features['release_to_add_lag'].dropna() < 0).sum())},
        {'check_name': 'valid_lag_titles', 'value': int(title_features['release_to_add_lag_clean'].notna().sum())},
        {'check_name': 'missing_country_titles', 'value': int(title_features['country_count'].eq(0).sum())},
        {'check_name': 'single_country_titles', 'value': int(title_features['country_count'].eq(1).sum())},
        {'check_name': 'multi_country_titles', 'value': int(title_features['country_count'].gt(1).sum())},
        {'check_name': 'country_view_rows', 'value': len(country_view)},
        {'check_name': 'genre_view_rows', 'value': len(genre_view)},
    ]
)

feature_distribution_summary = pd.DataFrame(
    [
        {
            'metric': 'country_count',
            'mean': title_features['country_count'].mean(),
            'median': title_features['country_count'].median(),
            'p75': title_features['country_count'].quantile(0.75),
            'max': title_features['country_count'].max(),
        },
        {
            'metric': 'genre_count',
            'mean': title_features['genre_count'].mean(),
            'median': title_features['genre_count'].median(),
            'p75': title_features['genre_count'].quantile(0.75),
            'max': title_features['genre_count'].max(),
        },
        {
            'metric': 'release_to_add_lag_clean',
            'mean': title_features['release_to_add_lag_clean'].mean(),
            'median': title_features['release_to_add_lag_clean'].median(),
            'p75': title_features['release_to_add_lag_clean'].quantile(0.75),
            'max': title_features['release_to_add_lag_clean'].max(),
        },
    ]
)

country_scope_summary = (
    title_features['country_scope']
    .value_counts(dropna=False)
    .rename_axis('country_scope')
    .reset_index(name='title_count')
)
country_scope_summary['share'] = country_scope_summary['title_count'] / country_scope_summary['title_count'].sum()

display(feature_qa)
display(feature_distribution_summary)
display(country_scope_summary)

,check_name,value
0,titles_row_count,6234
1,distinct_show_id,6234
2,negative_lag_titles,9
3,valid_lag_titles,6214
4,missing_country_titles,476
5,single_country_titles,4838
6,multi_country_titles,920
7,country_view_rows,7179
8,genre_view_rows,13670


,metric,mean,median,p75,max
0,country_count,1.152,1.000,1.000,12
1,genre_count,2.193,2.000,3.000,3
2,release_to_add_lag_clean,4.630,1.000,5.000,93


,country_scope,title_count,share
0,Single-country,4838,0.776
1,Multi-country,920,0.148
2,Missing country,476,0.076


**Interpretation**

- The title-level feature layer preserves full title coverage and keeps anomalies explicit rather than silently dropping them.
- Negative lag records are rare, which makes `release_to_add_lag_clean` a defensible business metric for freshness analysis.
- Most titles are single-country and multi-genre, which is exactly the structure the Phase 2 catalog and geographic analyses need.

## Bridge Views for Downstream Analysis

The final step is to project the engineered title-level features back onto the normalized entity views. This is how the business analysis notebook can answer questions like freshness by country or rating mix by genre without rebuilding transformations inline.

In [6]:
display(country_view[['show_id', 'country', 'type', 'country_scope', 'release_to_add_lag_clean']].head(8))
display(genre_view[['show_id', 'genre', 'type', 'rating_group', 'release_to_add_lag_clean']].head(8))

,show_id,country,type,country_scope,release_to_add_lag_clean
0,1005494,United States,Movie,Single-country,72
1,1008581,United States,Movie,Single-country,38
2,1065372,United States,Movie,Single-country,22
3,1067876,United States,Movie,Single-country,51
4,1145882,Hong Kong,Movie,Single-country,43
5,1150871,United States,Movie,Single-country,22
6,1151188,United States,Movie,Single-country,23
7,1151375,United States,Movie,Single-country,22


,show_id,genre,type,rating_group,release_to_add_lag_clean
0,1005494,Classic Movies,Movie,Family,72
1,1005494,Dramas,Movie,Family,72
2,1005494,Thrillers,Movie,Family,72
3,1008581,Classic Movies,Movie,Mature,38
4,1008581,Comedies,Movie,Mature,38
5,1008581,Cult Movies,Movie,Mature,38
6,1029730,Action & Adventure,Movie,Family,30
7,1029730,Comedies,Movie,Family,30


In [7]:
TABLES_DIR = ensure_directory(PROJECT_ROOT / 'outputs/tables')
save_dataframe(feature_dictionary, TABLES_DIR / 'phase2_feature_dictionary.csv')
save_dataframe(feature_qa, TABLES_DIR / 'phase2_feature_qa_summary.csv')
save_dataframe(feature_distribution_summary, TABLES_DIR / 'phase2_feature_distribution_summary.csv')
print('Saved feature engineering tables to', TABLES_DIR)

Saved feature engineering tables to /Users/xinyue/Documents/projects/netflix_da/outputs/tables


## Outcome

This notebook completes the missing intermediate layer in the repo structure:

- `01_data_audit.ipynb` validates the raw and processed data foundation
- `02_feature_engineering.ipynb` builds and validates the title-level feature layer
- `03_business_analysis.ipynb` consumes that layer for business-facing analysis

That separation makes the project easier to review and closer to a professional analytics workflow.